In [0]:
%python
from pyspark.sql.functions import expr ,col, current_timestamp,max
from pyspark.sql.types import DecimalType

class EodQuantityProcessor:
  def __init__(self):
    self.spark = spark


  def read_eod_stg_data(self):
    return (
              self.spark.read.table("dev.spark_db.eod_stg").where("as_of_dt = current_date()")
            )

  def get_eligible_data(self , eod_stg_df):

    eod_quantity_df = (
                      self.spark.read.table("dev.spark_db.eod_quantity")
                      )

    eod_qty_stg_df = (
                          eod_stg_df.alias("esd").join(
                                                      eod_quantity_df.alias("eqd"),
                                                      on = expr("""
                                                                  eqd.account_number = esd.account_number
                                                                  and
                                                                  eqd.asset_external_id = esd.asset_external_id
                                                                  """),
                                                      how = "left"
                                                  ).
                                                  select("esd.account_number",
                                                         "esd.asset_external_id",
                                                         "eqd.total_quantity",
                                                         "esd.total_qty",
                                                         col("esd.as_of_dt").alias("as_of_dt_esd"),
                                                         col("eqd.as_of_dt").alias("as_of_dt_eqd"),
                                                         "eqd.connection_id",
                                                         "eqd.eod_quantity_ver"
                                                         )
    )

    return eod_qty_stg_df

  def get_latest_row(self,eod_qty_stg_df):

    previous_account_row = (
                             eod_qty_stg_df.groupBy(col("account_number") , col("asset_external_id"))
                             .agg(
                                    max(col("as_of_dt_eqd")).alias("max_as_of_dt"),
                                    max(col("connection_id")).alias("max_connection_id")
                                  )

                            )

    detail_previous_row = ( eod_qty_stg_df.alias("es").join(
                                                      previous_account_row.alias("per"),
                                                      on = expr("""
                                                                es.account_number = per.account_number
                                                                and
                                                                es.asset_external_id = per.asset_external_id
                                                                and
                                                                es.as_of_dt_esd = per.max_as_of_dt
                                                                """),
                                                      how = "inner"
                                                      ).
                                                      select(
                                                          "es.account_number",
                                                          "es.asset_external_id",
                                                          "es.total_quantity",
                                                          col("max_as_of_dt").alias("as_of_dt"),
                                                          col("max_connection_id").alias("connection_id"),
                                                          "es.eod_quantity_ver"
                                                      )
    )

    return detail_previous_row


  def insert_eod_quantity(self,eod_stg_df , detail_of_previous_row):

    eod_insert_data = ( eod_stg_df.alias("esd").join(detail_of_previous_row.alias("prev"),
                                                    on = expr("""

                                                              esd.account_number = prev.account_number
                                                              and
                                                              esd.asset_external_id = prev.asset_external_id
                                                              """),
                                                    how = "left")
                                          .select("esd.account_number",
                                                  "esd.asset_external_id",
                                                  expr("esd.total_qty as quantity"),
                                                  "prev.total_quantity",
                                                  "esd.as_of_dt",
                                                  col("prev.as_of_dt").alias("as_of_dt_prev"),
                                                  "prev.connection_id",
                                                  "prev.eod_quantity_ver")
                    )
    eod_insert_data.display()

    eod_matching_insert = ( eod_insert_data.where("""
                                                    total_quantity is not null
                                                    and
                                                    as_of_dt_prev is not null
                                                    and
                                                    connection_id is not null
                                                  """)
                                          .select("account_number",
                                                  "asset_external_id",
                                                  "as_of_dt",
                                                  expr("total_quantity + quantity ").cast(DecimalType(18,2)).alias("total_quantity"),
                                                  expr("connection_id + 1 as connection_id"),
                                                  expr("eod_quantity_ver + 1 as eod_quantity_ver"),
                                                  current_timestamp().alias("eod_quantity_dml")
                                          )
    )
    


    eod_not_matching_insert = ( eod_insert_data.where("""
                                                    total_quantity is null
                                                    and 
                                                    as_of_dt_prev is null
                                                    and 
                                                    connection_id is null
                                                    """).
                                                    selectExpr("account_number",
                                                              "asset_external_id",
                                                              "as_of_dt",
                                                              "quantity as total_quantity",
                                                              "1 as connection_id",
                                                              "1 as eod_quantity_ver",
                                                              "current_timestamp() as eod_quantity_dml")
                    )
    

    insert_data = eod_matching_insert.union(eod_not_matching_insert)

    return insert_data


  def write_to_eod_quantity(self,eod_quantity_data):

    eod_quantity_data.write.format("delta")\
                            .mode("append")\
                            .saveAsTable("dev.spark_db.eod_quantity")

  def main(self):
    eod_stg_df = self.read_eod_stg_data()

    eligible_row = self.get_eligible_data(eod_stg_df)

    latest_row = self.get_latest_row(eligible_row)

    insert_data = self.insert_eod_quantity(eod_stg_df,latest_row)

    self.write_to_eod_quantity(insert_data)



if __name__ =='__main__':
  eod = EodQuantityProcessor()
  eod.main()




In [0]:
select * from dev.spark_db.eod_quantity;

In [0]:
select * from dev.spark_db.eod_quantity;

In [0]:
create or replace table dev.spark_db.eod_quantity(
    account_number string,
    asset_external_id string,
    as_of_dt date,
    total_quantity decimal(18,2),
    connection_id int,
    eod_quantity_ver int,
    eod_quantity_dml timestamp
)

In [0]:
select * from dev.spark_db.eod_stg;

In [0]:
drop table dev.spark_db.eod_quantity;